# 31. MCP (Model Context Protocol)

**Tier:** Production & Safety
**Estimated time:** 45 minutes
**Prerequisites:** 19, 30
**Priority:** 🟡 Important — MCP is the de-facto tool-integration standard and a common interview topic, but conceptually small once you know tool use (notebook 19). *If skipped, revisit when:* the first time you need to share tools across agents/clients, or integrate with Claude Desktop or another MCP host.
**Source material:** Anthropic's Model Context Protocol specification and official Python SDK; the sibling-protocol landscape (A2A/ACP/AG-UI/ANP) drawn from a community protocol-comparison reference

## What You'll Learn
- What MCP standardizes that raw tool-use (notebook 19) leaves every team to reinvent
- Building a small MCP server exposing tools over stdio using the official Python SDK
- Driving that server from a client in the same notebook
- Where MCP's access-scoping decisions (notebook 30) belong: at the server boundary, once, not per-agent
- How MCP relates to four sibling protocols — A2A, ACP, AG-UI, ANP — that standardize other edges of the agent stack, and when a production system needs more than one

## Why This Matters
Notebook 19's tool schemas work great for one agent, hand-written for one task. The moment you want to reuse the same tools across multiple agents, share them with a teammate's project, or plug into a host application (Claude Desktop, an IDE, another team's agent), you need a *protocol*, not just a Python dict convention — that's what MCP is.


In [1]:
import os, sys, logging, pathlib, tempfile

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

logging.getLogger("mcp").setLevel(logging.WARNING)   # quiet the server's default INFO logs

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"
print("Anthropic ready." if HAS_ANTHROPIC else "No ANTHROPIC_API_KEY — the LLM-driven cell will be skipped.")


Anthropic ready.


## What MCP standardizes

A tool schema dict (notebook 19) describes a tool to ONE model provider's SDK, wired into ONE agent's code. MCP standardizes three things raw tool-use leaves ad hoc:

1. **Discovery** — a client can ask any MCP server "what tools/resources do you expose?" without prior knowledge of that server's internals.
2. **Transport** — a consistent way to talk to a tool server, whether it's a local subprocess (stdio, what we use below) or a remote HTTP service — the client code doesn't change.
3. **A host-agnostic contract** — the same server works whether it's driven by your own agent code, Claude Desktop, an IDE extension, or a teammate's completely different agent framework.

The result: tools become a reusable service, not a block of dict literals copy-pasted into every new agent script.


## Where MCP sits among sibling protocols

MCP is one of five protocols that emerged through 2025–2026 to standardize a different edge of the agent stack. They are not competing alternatives to each other — each solves a different integration problem:

| Protocol | Standardizes | Concrete example |
|---|---|---|
| **MCP** — Model Context Protocol | Agent ↔ tools, data, APIs (what this notebook builds) | An agent calling `search_docs` or `calculator`, discovered at runtime |
| **A2A** — Agent2Agent | Agent ↔ another agent — discovery and task delegation across vendors/frameworks | Your LangGraph agent (notebook 20) handing a sub-task to a teammate's CrewAI agent, neither knowing the other's internals |
| **ACP** — Agent Communication Protocol | Agent ↔ an existing application's UI — operating software that was never built with agents in mind | An agent driving a legacy desktop tool through its screen, not an API, because no API exists |
| **AG-UI** — Agent-User Interaction Protocol | Agent ↔ a human, in real time | The token-by-token streaming and tool-call visualizations a chat interface needs from an agent backend |
| **ANP** — Agent Network Protocol | Agent ↔ the open internet of other agents | Discovering and collaborating with an agent you've never seen before, outside any single platform |

Read down that "standardizes" column and a pattern emerges: MCP governs what an agent can *use*. A2A governs what an agent can *hand off to another agent*. ACP governs what an agent can *operate* that has no API. AG-UI governs what a *human* sees while an agent works. ANP governs how an agent *finds* anyone else's agent at all. Five different edges of the same system, not five competing standards for one edge.

A real system increasingly combines more than one. A multi-agent research pipeline (notebook 20's patterns) might use MCP so each specialist agent reaches its own tools, A2A to delegate a sub-task to a different team's agent, and AG-UI to stream the whole run back to a person watching live.

**A caveat worth stating plainly:** MCP is by far the most mature of the five here — it has an official SDK, a large ecosystem, and is the one this notebook actually implements below. A2A, ACP, AG-UI, and ANP are newer, and as of this writing their specifications are still consolidating faster than any single notebook can track. Treat the table above as "what problem each one targets," not a claim that all four are equally standardized, equally adopted, or equally stable yet — that's genuinely still moving.


## Building an MCP server

`FastMCP` (the official SDK's high-level server helper) turns a plain Python function into an MCP tool with one decorator — the docstring and type hints become the tool's description and schema automatically, no hand-written JSON schema required (compare to notebook 19's `TOOL_SCHEMAS` list, written by hand).


In [2]:
SERVER_DIR = pathlib.Path(tempfile.mkdtemp(prefix="nb31_mcp_"))
# Note: tool descriptions are passed via the decorator's description= kwarg here (not a
# docstring) purely to avoid nesting triple-quoted strings inside this build script.
SERVER_SOURCE = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("notebook-tools")

_DOCS = {
    "kv cache": "KV caching stores past attention keys/values so decoding does not recompute them (see notebook 13).",
    "attention": "Attention lets a model weigh how relevant each token is to every other token (see notebook 4).",
}

@mcp.tool(description="Search internal course docs for a topic and return a summary.")
def search_docs(query: str) -> str:
    q = query.lower()
    hits = [v for k, v in _DOCS.items() if k in q]
    return " ".join(hits) if hits else "No results found."

@mcp.tool(description="Evaluate a basic arithmetic expression, e.g. 12 * 7.")
def calculator(expression: str) -> str:
    import re
    if not re.fullmatch(r"[0-9+*/(). -]+", expression):
        return "Error: invalid characters in expression."
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
(SERVER_DIR / "server.py").write_text(SERVER_SOURCE)
print(f"Wrote MCP server to {SERVER_DIR / 'server.py'}")


Wrote MCP server to /var/folders/61/c6m8f0q93sxdlrrn5gmplfbr0000gn/T/nb31_mcp_hskblkku/server.py


## Driving the server from a client

The client launches the server as a subprocess and speaks MCP over its stdin/stdout (the `stdio` transport) — no network port, no manual JSON-RPC framing, all handled by the SDK. First we discover what the server exposes, exactly as a host application like Claude Desktop would, without any prior knowledge of `server.py`'s internals.


In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_DIR / "server.py")])

async def list_server_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.list_tools()

tools_result = await list_server_tools()
for t in tools_result.tools:
    print(f"- {t.name}: {t.description}")


- search_docs: Search internal course docs for a topic and return a summary.
- calculator: Evaluate a basic arithmetic expression, e.g. 12 * 7.


## Calling tools through the protocol

Once discovered, calling a tool is a single `call_tool(name, arguments)` regardless of what the tool actually does inside the server — the exact same call shape notebook 19's raw loop hand-rolled per tool, now uniform across every MCP server you'll ever talk to.


In [4]:
async def call_mcp_tool(tool_name, arguments):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            return result.content[0].text

doc_answer = await call_mcp_tool("search_docs", {"query": "how does kv cache work"})
calc_answer = await call_mcp_tool("calculator", {"expression": "12 * 7"})
print("search_docs ->", doc_answer)
print("calculator  ->", calc_answer)


search_docs -> KV caching stores past attention keys/values so decoding does not recompute them (see notebook 13).
calculator  -> 84


## Wiring an MCP server into an agent loop

The payoff: an agent's tool-calling loop (notebook 19's shape) can source its tools from an MCP server instead of a hand-written dict, so the SAME server can be reused by this agent, a teammate's agent, or a completely different host application without any of them re-implementing `search_docs` or `calculator`.


In [5]:
async def run_agent_with_mcp(goal, max_steps=4):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    import anthropic
    client = anthropic.Anthropic()

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = await session.list_tools()
            # Translate MCP tool descriptors into Anthropic's tool schema format.
            anthropic_tools = [
                {"name": t.name, "description": t.description, "input_schema": t.inputSchema}
                for t in mcp_tools.tools
            ]
            messages = [{"role": "user", "content": goal}]
            for _ in range(max_steps):
                resp = client.messages.create(model=TEACH_MODEL, max_tokens=200,
                                               system="Use tools to answer precisely.",
                                               tools=anthropic_tools, messages=messages)
                if resp.stop_reason != "tool_use":
                    return " ".join(b.text for b in resp.content if b.type == "text")
                messages.append({"role": "assistant", "content": resp.content})
                results = []
                for b in resp.content:
                    if b.type == "tool_use":
                        call_result = await session.call_tool(b.name, b.input)
                        results.append({"type": "tool_result", "tool_use_id": b.id,
                                         "content": call_result.content[0].text})
                messages.append({"role": "user", "content": results})
    return "[budget exhausted]"

answer = await run_agent_with_mcp("What is 15 * 9? Use the calculator tool.")
print(answer)


15 * 9 = **135**


## Where notebook 30's access scoping belongs

MCP servers are the natural place to enforce the least-privilege and sandboxing defenses from notebook 30 — once, at the server boundary, instead of re-implementing them inside every agent that happens to use these tools. A server that only exposes `search_docs` and `calculator` structurally cannot be asked to read arbitrary files, regardless of how many different agents or hosts connect to it — the access decision is made in one place and every consumer inherits it.


## Exercises

**Exercise 1 (Warm-up):** Add a third tool, `word_count(text: str) -> int`, to `server.py`, restart the server (re-run the write-file cell), and confirm `list_server_tools()` now shows three tools.

**Exercise 2 (Apply):** Implement a small client-side cache: `cached_call_mcp_tool(tool_name, arguments)` that avoids re-launching the server subprocess for repeated identical calls within the same notebook run (hint: a dict keyed by `(tool_name, frozenset(arguments.items()))`).

**Exercise 3 (Extend):** Notebook 30 built `read_file_scoped` to enforce least privilege inside a single agent's Python code. Sketch how you'd move that same restriction into the MCP server itself (as a tool that only ever exposes a `public/` subdirectory) so EVERY client connecting to this server — not just this notebook's agent — inherits the restriction automatically.


In [6]:
# Exercise 1: Warm-up
# Task: Add word_count(text: str) -> int to SERVER_SOURCE, rewrite server.py, and re-list tools.
# Hint: follow the @mcp.tool() decorator pattern used by search_docs and calculator above.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement cached_call_mcp_tool(tool_name, arguments) with an in-memory cache dict.
# Hint: frozenset(arguments.items()) only works for flat, hashable argument values.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch adding a read_file tool to server.py that is hard-scoped to a public/ subdirectory,
# so every client of this server inherits the restriction (not just one agent's Python code).
# Hint: reuse notebook 30's read_file_scoped logic, but place it INSIDE the @mcp.tool() function.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
SERVER_SOURCE_V2 = SERVER_SOURCE.replace(
    'if __name__ == "__main__":',
    '''@mcp.tool(description="Count the number of words in the given text.")
def word_count(text: str) -> int:
    return len(text.split())

if __name__ == "__main__":'''
)
(SERVER_DIR / "server.py").write_text(SERVER_SOURCE_V2)
tools_v2 = await list_server_tools()
print([t.name for t in tools_v2.tools])

# Exercise 2
_tool_cache = {}
async def cached_call_mcp_tool(tool_name, arguments):
    key = (tool_name, frozenset(arguments.items()))
    if key not in _tool_cache:
        _tool_cache[key] = await call_mcp_tool(tool_name, arguments)
    return _tool_cache[key]

# Exercise 3
# Inside server.py, add:
'''
import pathlib
PUBLIC_DIR = pathlib.Path(__file__).parent / "public"
PUBLIC_DIR.mkdir(exist_ok=True)

@mcp.tool(description="Read a file, restricted to the server's public/ subdirectory.")
def read_file(path: str) -> str:
    target = (PUBLIC_DIR / path).resolve()
    if PUBLIC_DIR.resolve() not in target.parents and target != PUBLIC_DIR.resolve():
        return "Error: access denied outside public/."
    return target.read_text() if target.exists() else "Error: not found."
'''
# Now EVERY client — this notebook's agent, a teammate's agent, Claude Desktop — gets this
# restriction automatically, because it's enforced inside the tool implementation itself,
# not inside each agent's calling code.
```
</details>


## Key Takeaways
- MCP standardizes discovery, transport, and a host-agnostic contract for tools — the pieces raw tool-use (notebook 19) leaves every team to reinvent per project.
- `FastMCP` turns a typed, docstringed Python function into a fully-specified MCP tool with one decorator — no hand-written JSON schema.
- A client talks to any MCP server through the same three calls: connect, `list_tools()`, `call_tool()` — regardless of what's actually running inside that server.
- MCP is one of five emerging agent protocols, alongside A2A (agent-to-agent delegation), ACP (controlling apps with no API), AG-UI (real-time human-facing streaming), and ANP (open-network agent discovery) — each standardizes a different edge of the stack, and production systems increasingly combine more than one.
- Access-scoping and guardrails (notebook 30) belong INSIDE the MCP server, enforced once, so every client that connects inherits the restriction automatically instead of re-implementing it.
- In a Jupyter kernel, use top-level `await` for async MCP calls — `asyncio.run()` fails because ipykernel already runs its own event loop.

## What's Next
Notebook 32 covers cost engineering — model routing, prompt caching at scale, and batch economics — the discipline that keeps a served, tool-using agent affordable as usage grows.
